# Exploration 4 — Step 2: QUBO Solver + Binary Search

Runs the binary search algorithm over α for both SA and QA solvers.
Saves per-iteration logs and final summary to `data/`.

## 0. Imports & Paths

In [ ]:
import numpy as np
import pandas as pd
import openjij as oj
import time
from pathlib import Path

DATA = Path('data')
OUT  = Path('data')
print('OpenJij version:', oj.__version__)

## 1. Load Precomputed Data

In [ ]:
df_pairs      = pd.read_csv(DATA / 'pairs.csv')
df_importance = pd.read_csv(DATA / 'importance.csv')
df_redundancy = pd.read_csv(DATA / 'redundancy.csv', index_col=0)

fp_rss_train   = pd.read_csv(DATA / 'fp_rss_train.csv').values    # (K, L)
fp_rss_test    = pd.read_csv(DATA / 'fp_rss_test.csv').values     # (K, L)
fp_coords_train = pd.read_csv(DATA / 'fp_coords_train.csv').values # (K, 2)
fp_coords_test  = pd.read_csv(DATA / 'fp_coords_test.csv').values  # (K, 2)

I = df_importance['importance'].values   # (L,)  importance vector
R = df_redundancy.values                 # (L, L) redundancy matrix
pairs = list(zip(df_pairs['ap'], df_pairs['mp']))
L = len(pairs)
N = 10

print(f'Pairs (L)         : {L}')
print(f'Train fingerprints: {fp_rss_train.shape}')
print(f'Test  fingerprints: {fp_rss_test.shape}')

## 2. Parameters

In [ ]:
AP_BUDGET   = 5       # maximum distinct AP nodes
MP_BUDGET   = 5       # maximum distinct MP nodes
EPSILON     = 0.01    # binary search termination threshold (b - a < eps)
KNN_K       = 3       # number of neighbours in KNN localizer
NUM_READS   = 100     # solver reads per QUBO call
NUM_SWEEPS  = 1000    # sweeps per read
SEED        = 42

print(f'AP budget n={AP_BUDGET},  MP budget m={MP_BUDGET}')
print(f'Binary search epsilon={EPSILON}')

## 3. QUBO Builder

$$Q(\mathbf{x}, \alpha) = -\alpha \sum_p I_p x_p + (1-\alpha) \sum_{p \neq q} R_{pq} x_p x_q$$

OpenJij dict format:
- Linear  : `{(p, p): -α * I[p]}`
- Quadratic: `{(p, q): 2*(1-α)*R[p,q]}` for p < q  
  (factor 2 because Σ_{p≠q} = 2·Σ_{p<q} by symmetry of R)

In [ ]:
def build_qubo(alpha, I, R):
    """Return QUBO dict for a given alpha."""
    Q = {}
    L = len(I)
    # Linear terms (diagonal)
    for p in range(L):
        Q[(p, p)] = -alpha * I[p]
    # Quadratic terms (upper triangle, factor 2 for symmetry)
    coeff_scale = 2.0 * (1.0 - alpha)
    if coeff_scale != 0.0:
        for p in range(L):
            for q in range(p + 1, L):
                v = coeff_scale * R[p, q]
                if v != 0.0:
                    Q[(p, q)] = v
    return Q

# Quick sanity: build at alpha=0.5
_Q = build_qubo(0.5, I, R)
print(f'QUBO entries at alpha=0.5: {len(_Q)}  (expected ~{L + L*(L-1)//2})')

## 4. Helper: Extract AP/MP Counts from Solution

In [ ]:
def count_active(sample, pairs):
    """Return (AP_count, MP_count) for a solution dict {var_index: 0/1}."""
    active_ap = set()
    active_mp = set()
    for p, val in sample.items():
        if val == 1:
            i, j = pairs[p]
            active_ap.add(i)
            active_mp.add(j)
    return len(active_ap), len(active_mp)


def sample_to_vector(sample, L):
    """Convert solution dict to binary numpy array of length L."""
    x = np.zeros(L, dtype=int)
    for p, val in sample.items():
        x[p] = int(val)
    return x

## 5. KNN Localizer

In [ ]:
def localizer(x_vec, fp_rss_train, fp_coords_train,
              fp_rss_test, fp_coords_test, K=3):
    """
    KNN localizer using only the selected pairs.
    Returns mean Euclidean localization error in metres.
    """
    selected = np.where(x_vec == 1)[0]
    if len(selected) == 0:
        return np.inf

    train_feat = fp_rss_train[:, selected]   # (K_train, |selected|)
    test_feat  = fp_rss_test[:, selected]    # (K_test,  |selected|)

    errors = []
    for k in range(len(test_feat)):
        dists   = np.linalg.norm(train_feat - test_feat[k], axis=1)
        knn_idx = np.argsort(dists)[:K]
        pred    = fp_coords_train[knn_idx].mean(axis=0)
        true    = fp_coords_test[k]
        errors.append(np.linalg.norm(pred - true))

    return float(np.mean(errors))


# Sanity: baseline with ALL pairs selected
x_all = np.ones(L, dtype=int)
baseline_acc = localizer(x_all, fp_rss_train, fp_coords_train,
                          fp_rss_test,  fp_coords_test, K=KNN_K)
print(f'Baseline KNN error (all {L} pairs): {baseline_acc:.4f} m')

## 6. Binary Search

In [ ]:
def binary_search(sampler, sampler_name, I, R, pairs, L,
                  n_budget, m_budget, epsilon,
                  fp_rss_train, fp_coords_train,
                  fp_rss_test,  fp_coords_test,
                  knn_k=3, num_reads=100, num_sweeps=1000, seed=42):
    """
    Binary search on alpha.
    Search direction: budget only (AP_count <= n AND MP_count <= m).
    Best solution: tracked separately by KNN accuracy.
    Terminates when b - a < epsilon.
    """
    a, b       = 0.0, 1.0
    x_star     = None
    best_acc   = np.inf
    alpha_star = None
    log        = []

    iteration  = 0
    t_start    = time.time()

    while b - a >= epsilon:
        iteration += 1
        alpha = (a + b) / 2.0

        # Solve QUBO
        Q = build_qubo(alpha, I, R)
        response = sampler.sample_qubo(Q,
                                       num_reads=num_reads,
                                       num_sweeps=num_sweeps,
                                       seed=seed)
        best_sample = response.first.sample
        x_vec       = sample_to_vector(best_sample, L)

        ap_count, mp_count = count_active(best_sample, pairs)
        feasible = (ap_count <= n_budget) and (mp_count <= m_budget)

        acc = np.nan
        if feasible:
            a = alpha                           # push lower bound up
            acc = localizer(x_vec,
                            fp_rss_train, fp_coords_train,
                            fp_rss_test,  fp_coords_test,
                            K=knn_k)
            if acc < best_acc:                  # track best feasible solution
                best_acc   = acc
                x_star     = x_vec.copy()
                alpha_star = alpha
        else:
            b = alpha                           # push upper bound down

        log.append({
            'solver'    : sampler_name,
            'iteration' : iteration,
            'alpha'     : alpha,
            'a'         : a,
            'b'         : b,
            'AP_count'  : ap_count,
            'MP_count'  : mp_count,
            'feasible'  : feasible,
            'acc_m'     : acc,
            'pairs_selected': int(x_vec.sum()),
        })

        print(f'  [{sampler_name}] iter={iteration:2d}  alpha={alpha:.4f}  '
              f'AP={ap_count} MP={mp_count}  '
              f'{"FEASIBLE" if feasible else "OVER-BUDGET":12s}  '
              f'acc={acc:.4f} m' if feasible else
              f'  [{sampler_name}] iter={iteration:2d}  alpha={alpha:.4f}  '
              f'AP={ap_count} MP={mp_count}  OVER-BUDGET')

    t_total = time.time() - t_start

    return {
        'alpha_star' : alpha_star,
        'x_star'     : x_star,
        'best_acc'   : best_acc,
        'time_s'     : t_total,
        'iterations' : iteration,
        'log'        : log,
    }

## 7. Run SA (Simulated Annealing)

In [ ]:
print('=== Simulated Annealing ===')
sa_sampler = oj.SASampler()

sa_result = binary_search(
    sampler       = sa_sampler,
    sampler_name  = 'SA',
    I=I, R=R, pairs=pairs, L=L,
    n_budget      = AP_BUDGET,
    m_budget      = MP_BUDGET,
    epsilon       = EPSILON,
    fp_rss_train  = fp_rss_train,
    fp_coords_train = fp_coords_train,
    fp_rss_test   = fp_rss_test,
    fp_coords_test  = fp_coords_test,
    knn_k         = KNN_K,
    num_reads     = NUM_READS,
    num_sweeps    = NUM_SWEEPS,
    seed          = SEED,
)

print(f'\nSA  alpha*={sa_result["alpha_star"]:.4f}  '
      f'best_acc={sa_result["best_acc"]:.4f} m  '
      f'time={sa_result["time_s"]:.2f}s  '
      f'iters={sa_result["iterations"]}')

## 8. Run QA (Simulated Quantum Annealing)

In [ ]:
print('=== Simulated Quantum Annealing ===')
qa_sampler = oj.SQASampler()

qa_result = binary_search(
    sampler       = qa_sampler,
    sampler_name  = 'QA',
    I=I, R=R, pairs=pairs, L=L,
    n_budget      = AP_BUDGET,
    m_budget      = MP_BUDGET,
    epsilon       = EPSILON,
    fp_rss_train  = fp_rss_train,
    fp_coords_train = fp_coords_train,
    fp_rss_test   = fp_rss_test,
    fp_coords_test  = fp_coords_test,
    knn_k         = KNN_K,
    num_reads     = NUM_READS,
    num_sweeps    = NUM_SWEEPS,
    seed          = SEED,
)

print(f'\nQA  alpha*={qa_result["alpha_star"]:.4f}  '
      f'best_acc={qa_result["best_acc"]:.4f} m  '
      f'time={qa_result["time_s"]:.2f}s  '
      f'iters={qa_result["iterations"]}')

## 9. Save Results

In [ ]:
# --- Binary search logs (one row per iteration) ---
all_logs = sa_result['log'] + qa_result['log']
df_log = pd.DataFrame(all_logs)
df_log.to_csv(OUT / 'binary_search_log.csv', index=False)
print('Saved binary_search_log.csv')

# --- Selected pairs for each solver ---
for name, result in [('sa', sa_result), ('qa', qa_result)]:
    if result['x_star'] is not None:
        sel_idx = np.where(result['x_star'] == 1)[0]
        df_sel = df_pairs.iloc[sel_idx].copy()
        df_sel['importance'] = I[sel_idx]
        df_sel.to_csv(OUT / f'selected_pairs_{name}.csv', index=False)
        print(f'Saved selected_pairs_{name}.csv  ({len(sel_idx)} pairs)')

# --- Summary (one row per solver) ---
rows = []
for name, result in [('SA', sa_result), ('QA', qa_result)]:
    x = result['x_star']
    if x is not None:
        sample_dict = {p: int(x[p]) for p in range(L)}
        ap_c, mp_c = count_active(sample_dict, pairs)
    else:
        ap_c, mp_c = None, None
    rows.append({
        'solver'         : name,
        'alpha_star'     : result['alpha_star'],
        'AP_count'       : ap_c,
        'MP_count'       : mp_c,
        'pairs_selected' : int(x.sum()) if x is not None else None,
        'best_acc_m'     : result['best_acc'],
        'time_s'         : result['time_s'],
        'iterations'     : result['iterations'],
    })

df_summary = pd.DataFrame(rows)
df_summary.to_csv(OUT / 'summary.csv', index=False)

print('\nSummary:')
print(df_summary.to_string(index=False))